#### We have to find the 3rd highest total transaction amount from the records. We have two tables: one containing customer details (customers) and the other storing transaction data (card_orders).
- Our goal is to retrieve the customer who ranks third in terms of total transaction amount.



#### Solution
- To solve this problem using PySpark, we can break it down into 3 main steps:
    
- Aggregate the data: Join the customers and card_orders tables, then group the data by customer ID to calculate the total transaction amount for each customer.

- Rank the customers: Use the window function rank() to assign a rank to each customer based on their total transaction amount, in descending order.

- Filter the third-highest: Finally, filter the ranked data to retrieve the customer with the rank of 3 (i.e. the third-highest transaction total).

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Initialize the Spark session
spark = SparkSession.builder.appName("CustomerTransactions").getOrCreate()

In [0]:
# Create the customers DataFrame
customers_data = [
    (1, 'Jill', 'Doe', 'New York', '123 Main St', '555-1234'),
    (2, 'Henry', 'Smith', 'Los Angeles', '456 Oak Ave', '555-5678'),
    (3, 'William', 'Johnson', 'Chicago', '789 Pine Rd', '555-8765'),
    (4, 'Emma', 'Daniel', 'Houston', '321 Maple Dr', '555-4321'),
    (5, 'Charlie', 'Davis', 'Phoenix', '654 Elm St', '555-6789')
]
customers_columns = [
  'id', 
  'first_name', 
  'last_name', 
  'city', 
  'address', 
  'phone_number'
]

In [0]:
customers_df = spark.createDataFrame(customers_data, customers_columns)

In [0]:
# Create the card_orders DataFrame
card_orders_data = [
    (1, 1, '2024-11-01 10:00:00', 'Electronics', 200),
    (2, 2, '2024-11-02 11:30:00', 'Groceries', 150),
    (3, 1, '2024-11-03 15:45:00', 'Clothing', 120),
    (4, 3, '2024-11-04 09:10:00', 'Books', 90),
    (8, 3, '2024-11-08 10:20:00', 'Groceries', 130),
    (9, 1, '2024-11-09 12:00:00', 'Books', 180),
    (10, 4, '2024-11-10 11:15:00', 'Electronics', 200),
    (11, 5, '2024-11-11 14:45:00', 'Furniture', 150),
    (12, 2, '2024-11-12 09:30:00', 'Furniture', 180)
]
card_orders_columns = [
  'order_id', 
  'cust_id', 
  'order_date', 
  'order_details', 
  'total_order_cost'
]

In [0]:
card_orders_df = spark.createDataFrame(card_orders_data, card_orders_columns)

In [0]:
# Join the customers and card_orders DataFrames to calculate the total transaction amount
customer_transactions = customers_df.join(card_orders_df, customers_df.id == card_orders_df.cust_id) \
    .groupBy(customers_df.id, customers_df.first_name, customers_df.last_name) \
    .agg(F.sum(card_orders_df.total_order_cost).alias('total_transaction_amount'))


In [0]:
# Rank the customers based on their total transaction amount
window_spec = Window.orderBy(F.desc('total_transaction_amount'))
ranked_transactions = customer_transactions.withColumn('rank', F.rank().over(window_spec))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Select the customer with the third-highest total transaction amount
third_highest_customer = ranked_transactions.filter(ranked_transactions.rank == 3) \
    .select('id', 'first_name', 'last_name')
third_highest_customer.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  3|   William|  Johnson|
+---+----------+---------+

